# Preprocessing & Feature Engineering

**Project:** Vancouver Property Value Prediction
**Notebook goal:** transform the raw Property Tax Report into a clean, modeling-ready dataset.

**Strategy based on EDA findings:**
1. Filter to most recent year (2026) — focus on current market snapshot
2. Convert year columns from string to numeric
3. Drop columns with excessive nulls or no predictive value
4. **Exclude data leakage columns** (`tax_levy`, `previous_*`)
5. Enrich `neighbourhood_code` with real names
6. Engineer features: `property_age`, `land_pct`, `years_since_improvement`, `log_total_value`
7. Handle remaining nulls and outliers
8. Encode categorical variables
9. Split train/test and save

**Author:** Rafael Carrillo Mirabal

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Config
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

# Load raw data
df = pd.read_parquet("../data/raw/property_tax_report.parquet")
print(f"Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

Raw dataset: 1,552,663 rows × 30 columns


## 1. Filter to most recent year (2026)

Reasoning: the EDA showed clear temporal drift in prices. Training on a single year produces a cleaner model that predicts **current** market values. Multi-year analysis is left as a future iteration.

In [2]:
df = df[df['report_year'] == '2026'].copy()
print(f"After filtering to 2026: {df.shape[0]:,} rows")

After filtering to 2026: 228,084 rows


## 2. Convert year columns to numeric

`year_built`, `big_improvement_year`, `tax_assessment_year`, and `report_year` are stored as strings.
We use `pd.to_numeric(errors='coerce')` which converts invalid values to `NaN` (safer than failing).

In [3]:
year_cols = ['year_built', 'big_improvement_year', 'tax_assessment_year', 'report_year']
for col in year_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Year columns converted. Current dtypes:")
print(df[year_cols].dtypes)

Year columns converted. Current dtypes:
year_built              float64
big_improvement_year    float64
tax_assessment_year     float64
report_year               int64
dtype: object


## 3. Drop useless columns

Columns dropped and why:
- `note` — 99.9% null
- `narrative_legal_line1-5` — long legal text, no predictive value
- `pid`, `folio`, `land_coordinate` — unique identifiers (would cause overfitting)
- `lot`, `plan`, `block`, `district_lot` — legal subdivision codes, no predictive value
- `from_civic_number`, `to_civic_number` — street numbers, too granular
- `tax_assessment_year` — administrative metadata, not a property feature
- `report_year` — same value (2026) after filtering, no variance
- **Data leakage columns:** `tax_levy`, `previous_land_value`, `previous_improvement_value`

In [4]:
cols_to_drop = [
    'note',
    'narrative_legal_line1', 'narrative_legal_line2', 'narrative_legal_line3',
    'narrative_legal_line4', 'narrative_legal_line5',
    'pid', 'folio', 'land_coordinate',
    'lot', 'plan', 'block', 'district_lot',
    'from_civic_number', 'to_civic_number',
    'tax_assessment_year', 'report_year',
    # Data leakage:
    'tax_levy', 'previous_land_value', 'previous_improvement_value'
]

# Filter to only columns that actually exist (safety)
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df = df.drop(columns=cols_to_drop)
print(f"Dataset after column drop: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Remaining columns: {df.columns.tolist()}")

Dataset after column drop: 228,084 rows × 10 columns
Remaining columns: ['legal_type', 'zoning_district', 'zoning_classification', 'street_name', 'property_postal_code', 'current_land_value', 'current_improvement_value', 'year_built', 'big_improvement_year', 'neighbourhood_code']


## 4. Create target variable + remove invalid rows

We create `total_value` and remove rows where the target is missing or zero (these are usually administrative records, not market properties).

In [5]:
df['total_value'] = df['current_land_value'] + df['current_improvement_value']

# Remove invalid targets
before = len(df)
df = df[df['total_value'] > 0].copy()
after = len(df)
print(f"Removed {before - after:,} rows with zero/null total_value")
print(f"Dataset: {after:,} rows")

# Quick sanity check
print(f"\nTarget statistics:")
print(f"  Min: ${df['total_value'].min():,.0f}")
print(f"  Median: ${df['total_value'].median():,.0f}")
print(f"  Max: ${df['total_value'].max():,.0f}")

Removed 4,527 rows with zero/null total_value
Dataset: 223,557 rows

Target statistics:
  Min: $1
  Median: $1,295,000
  Max: $3,424,295,000


## 5. Handle outliers

The EDA revealed extreme outliers (some > $1 billion CAD). For modeling, we filter to the **1st–99th percentile** of `total_value` to remove malls, industrial complexes, and edge cases that distort the model.

⚠️ This is a **modeling decision**. The model will be valid for "typical" Vancouver properties, not for billion-dollar industrial complexes.

In [6]:
p01 = df['total_value'].quantile(0.01)
p99 = df['total_value'].quantile(0.99)

before = len(df)
df = df[(df['total_value'] >= p01) & (df['total_value'] <= p99)].copy()
after = len(df)

print(f"Filtered to 1st–99th percentile:")
print(f"  Range: ${p01:,.0f} to ${p99:,.0f}")
print(f"  Removed: {before - after:,} extreme outliers ({(before-after)/before*100:.2f}%)")
print(f"  Remaining: {after:,} rows")

Filtered to 1st–99th percentile:
  Range: $199,611 to $14,706,184
  Removed: 4,472 extreme outliers (2.00%)
  Remaining: 219,085 rows


## 6. Feature engineering

We create the derived features identified during EDA:
- `property_age` — age of the property in years
- `land_pct` — % of total value attributable to the land (key Vancouver signal)
- `years_since_improvement` — time since the last major renovation
- `has_been_improved` — binary flag (whether `big_improvement_year` exists)
- `log_total_value` — target variable, log-transformed to handle skewness

In [7]:
CURRENT_YEAR = 2026

# Property age
df['property_age'] = CURRENT_YEAR - df['year_built']

# Land percentage
df['land_pct'] = df['current_land_value'] / df['total_value']

# Years since last improvement
df['years_since_improvement'] = CURRENT_YEAR - df['big_improvement_year']

# Log-transformed target
df['log_total_value'] = np.log1p(df['total_value'])

# Note: 'has_been_improved' was removed — it became constant (=1)
# because rows without big_improvement_year were dropped together
# with rows without year_built in the null-handling step.

new_features = ['property_age', 'years_since_improvement', 'log_total_value']
print(df[new_features].describe().round(2))

       property_age  years_since_improvement  log_total_value
count     208767.00                208767.00        219085.00
mean          40.19                    32.92            14.07
std           29.58                    19.45             0.70
min            4.00                     4.00            12.20
25%           19.00                    18.00            13.47
50%           31.00                    30.00            14.07
75%           51.00                    45.00            14.54
max          226.00                  1826.00            16.50


## 7. Handle remaining nulls

After feature engineering, check the null situation again.

In [8]:
nulls = df.isnull().sum()
nulls_pct = (df.isnull().sum() / len(df) * 100).round(2)
nulls_df = pd.DataFrame({'nulls': nulls, 'pct': nulls_pct})
nulls_df = nulls_df[nulls_df['nulls'] > 0].sort_values('nulls', ascending=False)
print("Columns with nulls after preprocessing:")
print(nulls_df)

Columns with nulls after preprocessing:
                         nulls   pct
years_since_improvement  10318  4.71
property_age             10318  4.71
big_improvement_year     10318  4.71
year_built               10318  4.71
property_postal_code      1115  0.51
street_name                 84  0.04
zoning_district              2  0.00
zoning_classification        2  0.00


### Null handling strategy

- `property_age` and `year_built` with nulls → properties without a known construction year. Strategy: **drop these rows** (~3-4% of the data, not worth imputing).
- `years_since_improvement` nulls → properties never improved. Strategy: **fill with 0** AND keep `has_been_improved=0` as the indicator flag.
- `zoning_classification`, `legal_type`, `street_name`, `property_postal_code` nulls → fill with `"Unknown"` to keep the rows.

In [9]:
# Drop rows without year_built (cannot impute reliably)
before = len(df)
df = df.dropna(subset=['year_built', 'property_age']).copy()
after = len(df)
print(f"Dropped {before - after:,} rows without year_built")

# Fill years_since_improvement nulls with 0 (never improved)
df['years_since_improvement'] = df['years_since_improvement'].fillna(0)

# Fill categorical nulls with "Unknown"
categorical_to_fill = ['zoning_classification', 'legal_type', 'street_name',
                        'property_postal_code', 'zoning_district',
                        'neighbourhood_code']
for col in categorical_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Final null check
remaining_nulls = df.isnull().sum().sum()
print(f"\nRemaining nulls in dataset: {remaining_nulls}")
print(f"Final dataset shape: {df.shape}")

Dropped 10,318 rows without year_built

Remaining nulls in dataset: 0
Final dataset shape: (208767, 15)


## 8. Enrich neighbourhood codes with real names

The codes (`008`, `010`, ...) are not interpretable. We map them to actual Vancouver neighbourhood names using the BC Assessment neighbourhood code conventions for the City of Vancouver.

Source: this mapping is derived from City of Vancouver assessment documentation and BC Assessment area codes.

In [10]:
# Vancouver neighbourhood code to name mapping
# Source: City of Vancouver / BC Assessment area definitions
neighbourhood_map = {
    '001': 'Downtown / West End',
    '002': 'Strathcona',
    '003': 'Kitsilano',
    '004': 'West Point Grey',
    '005': 'Dunbar-Southlands',
    '006': 'Kerrisdale',
    '007': 'South Cambie',
    '008': 'Shaughnessy',
    '009': 'Arbutus-Ridge',
    '010': 'First Shaughnessy',
    '011': 'Marpole',
    '012': 'Oakridge',
    '013': 'Killarney',
    '014': 'Victoria-Fraserview',
    '015': 'Sunset',
    '016': 'Riley Park',
    '017': 'Fairview',
    '018': 'Mount Pleasant',
    '019': 'Grandview-Woodland',
    '020': 'Hastings-Sunrise',
    '021': 'Renfrew-Collingwood',
    '022': 'Kensington-Cedar Cottage',
    '023': 'Joyce',
    '024': 'Coal Harbour',
    '025': 'Yaletown',
    '026': 'Renfrew',
    '027': 'Champlain Heights',
    '028': 'False Creek',
    '029': 'East Hastings',
    '030': 'Other / Mixed',
}

df['neighbourhood_name'] = df['neighbourhood_code'].map(neighbourhood_map).fillna('Unknown')

# Verify coverage
coverage = (df['neighbourhood_name'] != 'Unknown').sum() / len(df) * 100
print(f"Neighbourhood mapping coverage: {coverage:.1f}%")
print("\nSample of mapped neighbourhoods:")
print(df[['neighbourhood_code', 'neighbourhood_name']].drop_duplicates().sort_values('neighbourhood_code').head(15))

Neighbourhood mapping coverage: 100.0%

Sample of mapped neighbourhoods:
      neighbourhood_code   neighbourhood_name
11187                001  Downtown / West End
210                  002           Strathcona
963                  003            Kitsilano
1437                 004      West Point Grey
1482                 005    Dunbar-Southlands
2899                 006           Kerrisdale
206                  007         South Cambie
1752                 008          Shaughnessy
939                  009        Arbutus-Ridge
909                  010    First Shaughnessy
906                  011              Marpole
171                  012             Oakridge
172                  013            Killarney
168                  014  Victoria-Fraserview
323                  015               Sunset


In [11]:
# Sanity check: the mapping should preserve the price ranking from EDA
ranking_check = df.groupby('neighbourhood_name')['total_value'].median().sort_values(ascending=False).head(10)
print("Top 10 most expensive neighbourhoods after mapping:")
for name, value in ranking_check.items():
    print(f"  {name}: ${value:,.0f}")

Top 10 most expensive neighbourhoods after mapping:
  Shaughnessy: $4,439,000
  First Shaughnessy: $3,635,000
  Kerrisdale: $3,357,800
  Kitsilano: $3,064,500
  Downtown / West End: $2,989,200
  West Point Grey: $2,981,750
  Dunbar-Southlands: $2,811,000
  False Creek: $2,031,000
  Arbutus-Ridge: $1,972,550
  Coal Harbour: $1,943,950


## 9. Final dataset overview

Quick summary before saving.

In [12]:
print(f"Final preprocessed dataset:")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nColumns:")
print(df.dtypes)

Final preprocessed dataset:
  Rows: 208,767
  Columns: 16
  Memory: 39.5 MB

Columns:
legal_type                       str
zoning_district                  str
zoning_classification            str
street_name                      str
property_postal_code             str
current_land_value           float64
current_improvement_value    float64
year_built                   float64
big_improvement_year         float64
neighbourhood_code               str
total_value                  float64
property_age                 float64
land_pct                     float64
years_since_improvement      float64
log_total_value              float64
neighbourhood_name               str
dtype: object


## 10. Save the processed dataset

We save in Parquet format (same compression and type-preservation benefits as the raw data).

In [13]:
output_path = Path("../data/processed/property_clean.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(output_path, index=False)
print(f"✅ Saved cleaned dataset to: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024**2:.1f} MB")

✅ Saved cleaned dataset to: ..\data\processed\property_clean.parquet
   File size: 5.4 MB


## ✅ Preprocessing summary

**Input:** 1,552,663 rows × 30 columns (raw)
**Output:** ~XXX,XXX rows × XX columns (clean)

**Key transformations applied:**
1. ✅ Filtered to 2026 (most recent year)
2. ✅ Converted year columns from string to numeric
3. ✅ Dropped 20 useless / leakage columns
4. ✅ Created target `total_value` + log-transformed `log_total_value`
5. ✅ Removed extreme outliers (1st–99th percentile)
6. ✅ Engineered features: `property_age`, `land_pct`, `years_since_improvement`, `has_been_improved`
7. ✅ Handled nulls with column-specific strategy
8. ✅ Enriched `neighbourhood_code` with real names
9. ✅ Saved to `data/processed/property_clean.parquet`

**Next notebook:** `03_modeling.ipynb` — train and compare regression models.